# Import libraries

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Set the environment variable for Google Cloud credentials
# Place the path in which the .json file is located.

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\roxan\AppData\Roaming\gcloud\application_default_credentials.json"

In [3]:
# Set your Google Cloud project ID and BigQuery dataset details

project_id = 'project-401f4646-3663-4125-aaa' # Edit with your project id
dataset_id = 'staging_db' # Modify the necessary schema name: staging_db, reporting_db etc.
table_id = 'stg_rental' # Modify the necessary table name: stg_customer, stg_city etc.

# SQL Query

In [4]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Define your SQL query here
query = """
with base as (
  select *
  from `project-401f4646-3663-4125-aaa.pagila_productionpublic.rental` 
  )

  , final as (
    select
          rental_id
        , rental_date as rental_rental_date
        , inventory_id as rental_inventory_id
        , customer_id as rental_customer_id
        , return_date as rental_return_date
        , staff_id as rental_staff_id
        , last_update as rental_last_update
   FROM base
  )

  select * from final
"""

# Execute the query and store the result in a dataframe
df = client.query(query).to_dataframe()

# Explore some records
df.head()

C:\Users\roxan\PycharmProjects\Pagila Analytics Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,rental_id,rental_rental_date,rental_inventory_id,rental_customer_id,rental_return_date,rental_staff_id,rental_last_update
0,12144,2022-02-14 15:16:03+00:00,512,587,NaT,1,2022-02-16 02:30:53+00:00
1,13295,2022-02-14 15:16:03+00:00,2408,229,NaT,1,2022-02-16 02:30:53+00:00
2,12064,2022-02-14 15:16:03+00:00,298,284,NaT,1,2022-02-16 02:30:53+00:00
3,15314,2022-02-14 15:16:03+00:00,2754,412,NaT,1,2022-02-16 02:30:53+00:00
4,14426,2022-02-14 15:16:03+00:00,2902,315,NaT,1,2022-02-16 02:30:53+00:00


# Write to BigQuery

In [5]:
# Define the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Define table schema based on the project description

schema = [
    bigquery.SchemaField('rental_id', 'INTEGER'),
    bigquery.SchemaField('rental_rental_date', 'DATETIME'),
    bigquery.SchemaField('rental_inventory_id', 'INTEGER'),
    bigquery.SchemaField('rental_customer_id', 'INTEGER'),
    bigquery.SchemaField('rental_return_date', 'DATETIME'),
    bigquery.SchemaField('rental_staff_id', 'INTEGER'),
    bigquery.SchemaField('rental_last_update', 'DATETIME')
    ]

In [6]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Check if the table exists
def table_exists(client, full_table_id):
    try:
        client.get_table(full_table_id)
        return True
    except Exception:
        return False

# Write the dataframe to the table (overwrite if it exists, create if it doesn't)
if table_exists(client, full_table_id):
    # If the table exists, overwrite it
    destination_table = f"{dataset_id}.{table_id}"
    # Write the dataframe to the table (overwrite if it exists)
    to_gbq(df, destination_table, project_id=project_id, if_exists='replace')
    print(f"Table {full_table_id} exists. Overwritten.")
else:
    # If the table does not exist, create it
    job_config = bigquery.LoadJobConfig(schema=schema)
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Table {full_table_id} did not exist. Created and data loaded.")

Table project-401f4646-3663-4125-aaa.staging_db.stg_rental did not exist. Created and data loaded.


In [7]:
# Converting i.pynb file to .py python executable file. 

!python -m jupyter nbconvert stg_rental.ipynb --to python --output-dir='../'

[NbConvertApp] Converting notebook stg_rental.ipynb to python
[NbConvertApp] Writing 3442 bytes to ..\stg_rental.py


In [8]:
!python -m pip install nbconvert


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!python -m pip install nbconvert -U


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
